# 講師用: キャッシュデータ準備ノートブック
## Qwen3-8B 感情ベクトル講義のための事前計算

このノートブックは、**学生用 `lecture_notebook.ipynb` で使うキャッシュデータを生成する** ためのものです。

### 何を生成するのか

1. `stories_smoke.jsonl` ― Qwen3-8B が生成した 280 個のストーリー (14 感情 × 20 個)
2. `activations_smoke.npz` ― 上記ストーリーから抽出した隠れ状態 (層 24)
3. `emotion_vectors_smoke.npz` ― 14 個の最終感情ベクトル

### 実行環境

- **Google Colab T4 GPU** ランタイム (推奨)
- ローカル GPU でも動きますが、`bitsandbytes` の 4-bit 量子化を使うので CUDA が必要です
- M1/M2/M4 Mac では 4-bit 量子化が動かないため、Colab を使ってください

### 所要時間

- ストーリー生成: 約 30-45 分
- 活性化抽出: 約 5-10 分
- 合計: **45-60 分** (T4 GPU で)

### 学期に 1 回だけ実行すれば OK

生成したキャッシュは GitHub にアップロードします。学生のノートブックは GitHub から自動でダウンロードします。


---

## ステップ 1: 環境セットアップ


In [ ]:
import subprocess, sys, os, gc, json, re
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

PACKAGES = [
    "transformers>=4.51.0,<5.0.0",
    "huggingface_hub<1.0.0",
    "tokenizers>=0.21.0,<0.22.0",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "scikit-learn",
    "pandas",
    "tqdm",
    "sentencepiece",
]

if IN_COLAB:
    print("Colab を検出。ライブラリをインストールしています...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    print("✅ インストール完了")

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "T4 GPU を有効にしてください (ランタイム → ランタイムのタイプを変更)"

# 出力ディレクトリ
OUTPUT_DIR = Path("/content/cache_to_upload") if IN_COLAB else Path.cwd() / "cache_to_upload"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CACHE = Path("/content/models") if IN_COLAB else Path.cwd() / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"出力ディレクトリ: {OUTPUT_DIR}")


---

## ステップ 2: 設定

ここで生成するデータの規模と感情リストを設定します。デフォルト値は学生のノートブックに合うように設定済みです。


In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"

# 学生ノートブックの DEFAULT_STUDY_EMOTIONS に対応
SELECTED_EMOTIONS = [
    "happy", "sad", "angry", "afraid", "calm", "loving", "hostile", "desperate",
    "surprised", "proud", "gloomy", "reflective", "blissful", "enthusiastic",
]

# 各感情あたりのストーリー数 = TOPICS_PER_EMOTION × STORIES_PER_TOPIC = 20
TOPICS_PER_EMOTION = 5
STORIES_PER_TOPIC = 4

# 中間層 = 36 層中 24 層目 (66% の位置)
# 学生ノートブックの LAYER_TO_PROBE と一致
LAYER_TO_PROBE_FRAC = 2 / 3

# 各種出力ファイル
STORIES_PATH = OUTPUT_DIR / "stories_smoke.jsonl"
ACTIVATIONS_PATH = OUTPUT_DIR / "activations_smoke.npz"
EMOTION_VECTORS_PATH = OUTPUT_DIR / "emotion_vectors_smoke.npz"

print(f"感情数: {len(SELECTED_EMOTIONS)}")
print(f"目標ストーリー数: {len(SELECTED_EMOTIONS)} × {TOPICS_PER_EMOTION} × {STORIES_PER_TOPIC} = {len(SELECTED_EMOTIONS) * TOPICS_PER_EMOTION * STORIES_PER_TOPIC}")


---

## ステップ 3: ストーリー生成のためのトピックバンク

感情語そのものを使わずに感情を表現するための、**ニュートラルなトピック** のリストです。元のノートブックから抜粋しています。


In [ ]:
TOPIC_BANK = [
    "a worker is asked to train the person who may replace them",
    "a traveler misses a major event because of a delayed flight",
    "two friends remember the same event in incompatible ways",
    "a student is accused of plagiarism right before graduation",
    "someone finds a hidden room in a new house",
    "a partner has been secretly learning the other person's native language",
    "a family heirloom turns up in a pawn shop",
    "a recipe becomes famous under someone else's name",
    "someone receives a message from a childhood bully after many years",
    "a coach has to cut a player they deeply believe in",
    "a childhood diary is published online without permission",
    "a favorite neighborhood restaurant is closing",
    "two strangers realize they have been dating the same person",
    "an athlete is told to switch positions at the last minute",
    "a person discovers they were adopted through a DNA test",
    "a junior coworker is being paid more than a senior employee",
    "someone finds a wallet filled with cash and an unfinished letter",
    "a mentor retires without warning or goodbye",
    "an invention is already patented by someone else",
    "a student loses a scholarship they counted on",
]

SELECTED_TOPICS = TOPIC_BANK[:TOPICS_PER_EMOTION]
print(f"使用するトピック ({TOPICS_PER_EMOTION} 個):")
for i, t in enumerate(SELECTED_TOPICS, 1):
    print(f"  {i}. {t}")


---

## ステップ 4: Qwen3-8B を 4-bit 量子化で読み込む

VRAM に約 6GB を使います。所要時間: 2-3 分。


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_CACHE,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

NUM_LAYERS = len(model.model.layers)
HIDDEN_SIZE = model.config.hidden_size
LAYER_TO_PROBE = int(round(NUM_LAYERS * LAYER_TO_PROBE_FRAC))

print(f"\n✅ モデル読み込み完了")
print(f"  レイヤー数: {NUM_LAYERS}")
print(f"  隠れ次元: {HIDDEN_SIZE}")
print(f"  対象レイヤー: {LAYER_TO_PROBE}")
print(f"  使用 VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


---

## ステップ 5: ストーリーを生成する

各感情 × 各トピックの組み合わせで Qwen3-8B にストーリーを書かせます。

**所要時間: 約 30-45 分**

途中で停止しても、`STORIES_PATH` に既に保存された分は次回スキップされるので、再開できます。


In [ ]:
def build_story_prompt(topic, emotion, n_stories):
    return f"""
Write {n_stories} short, unrelated stories around this topic:

Topic: {topic}

Each story should make it obvious that a focal character is feeling {emotion}, but do not name the emotion directly and do not rely on obvious synonyms.

Show the emotion indirectly through:
- actions
- body language
- internal thoughts
- dialogue
- the surrounding situation

Mix first-person and third-person narration across the stories.
Format the output as:
[story 1]
...

[story 2]
...
"""


def split_bracketed(text, label="story"):
    """[story 1] / [story 2] のような区切りで分割"""
    pattern = re.compile(rf"\[\s*{label}\s*\d+\s*\]", flags=re.IGNORECASE)
    chunks = [c.strip() for c in pattern.split(text) if c.strip()]
    if not chunks:
        chunks = [c.strip() for c in text.split("\n\n") if len(c.split()) > 20]
    return chunks


def generate_text(messages, max_new_tokens=420, temperature=0.8, top_p=0.9, top_k=20):
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def load_jsonl(path):
    if not Path(path).exists():
        return []
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


# 既存のキャッシュをロード (途中再開用)
existing = load_jsonl(STORIES_PATH)
existing_keys = {(r["emotion"], r["topic"]) for r in existing}
print(f"既存のストーリー: {len(existing)} 個")

# 不足分のみ生成
total_combos = len(SELECTED_EMOTIONS) * len(SELECTED_TOPICS)
done = 0

for emotion in tqdm(SELECTED_EMOTIONS, desc="感情"):
    for topic in SELECTED_TOPICS:
        if (emotion, topic) in existing_keys:
            done += 1
            continue

        messages = [
            {"role": "system", "content": "You write vivid but concise fiction."},
            {"role": "user", "content": build_story_prompt(topic, emotion, STORIES_PER_TOPIC)},
        ]
        try:
            generated = generate_text(messages, max_new_tokens=420)
            stories = split_bracketed(generated, "story")[:STORIES_PER_TOPIC]
        except Exception as e:
            print(f"  Error for {emotion} / {topic}: {e}")
            continue

        new_rows = [
            {"emotion": emotion, "topic": topic, "story_idx": i, "text": s}
            for i, s in enumerate(stories)
        ]
        append_jsonl(STORIES_PATH, new_rows)
        existing_keys.add((emotion, topic))
        done += 1

stories_df = pd.DataFrame(load_jsonl(STORIES_PATH))
print(f"\n✅ 全ストーリー: {len(stories_df)} 個")
print(stories_df.groupby("emotion").size().rename("count").to_frame().T)


---

## ステップ 6: 隠れ状態を抽出する

各ストーリーを Qwen3-8B に流し、**24 層目** の隠れ状態を取り出します。所要時間: 約 5-10 分。

学生ノートブックと同じ pooling 戦略 (最初の 4 トークンを除いた残りトークンの平均) を使います。


In [ ]:
def extract_story_activations(texts, layer_idx, batch_size=4, pool_skip_first=4):
    model.model.layers[layer_idx]._forward_hooks.clear()
    captured = {}

    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["seq"] = hidden.detach().clone()

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    pooled_list = []
    tokenizer.padding_side = "right"

    for i in tqdm(range(0, len(texts), batch_size), desc="活性化抽出"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            _ = model(**inputs)
        hidden = captured["seq"].float().cpu()
        attention_mask = inputs["attention_mask"].cpu()

        for row in range(hidden.shape[0]):
            seq_len = int(attention_mask[row].sum().item())
            start = min(pool_skip_first, seq_len - 1)
            pooled = hidden[row, start:seq_len].mean(dim=0).numpy()
            pooled_list.append(pooled)

    handle.remove()
    return np.stack(pooled_list)


story_matrix = extract_story_activations(
    stories_df["text"].tolist(),
    layer_idx=LAYER_TO_PROBE,
    batch_size=4,
)

print(f"\n活性化 shape: {story_matrix.shape}")
print(f"格納形式: float16 ({story_matrix.astype(np.float16).nbytes / 1024 / 1024:.2f} MB)")

# float16 で保存 (容量を半分に)
np.savez_compressed(
    ACTIVATIONS_PATH,
    activations=story_matrix.astype(np.float16),
    emotions=stories_df["emotion"].to_numpy(),
)
print(f"✅ 保存完了: {ACTIVATIONS_PATH}")
print(f"  ファイルサイズ: {ACTIVATIONS_PATH.stat().st_size / 1024 / 1024:.2f} MB")


---

## ステップ 7: 感情ベクトルを計算する

Difference of Means 法で 14 個の感情ベクトルを計算します。


In [ ]:
story_labels = stories_df["emotion"].to_numpy()
ordered_emotions = sorted(stories_df["emotion"].unique())

emotion_means = []
for emo in ordered_emotions:
    mask = (story_labels == emo)
    emotion_means.append(story_matrix[mask].mean(axis=0))
emotion_means = np.stack(emotion_means)

global_center = emotion_means.mean(axis=0)
emotion_vectors = emotion_means - global_center

np.savez_compressed(
    EMOTION_VECTORS_PATH,
    vectors=emotion_vectors.astype(np.float16),
    emotions=np.array(ordered_emotions),
    global_center=global_center.astype(np.float16),
)

print(f"✅ {len(ordered_emotions)} 個の感情ベクトルを保存しました")
print(f"  パス: {EMOTION_VECTORS_PATH}")
print(f"  shape: {emotion_vectors.shape}")
print(f"  ファイルサイズ: {EMOTION_VECTORS_PATH.stat().st_size / 1024:.1f} KB")


---

## ステップ 8: 動作確認 (任意)

生成したベクトルが妥当かどうかを、コサイン類似度で軽くチェックします。`happy` ↔ `sad` は負、`angry` ↔ `hostile` は正になるはずです。


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

vec_dict = {emo: emotion_vectors[i] for i, emo in enumerate(ordered_emotions)}

pairs_to_check = [
    ("happy", "sad",       "negative"),  # 反対方向
    ("happy", "blissful",  "positive"),  # 近い
    ("angry", "hostile",   "positive"),  # 近い
    ("calm",  "desperate", "negative"),  # 反対方向
    ("sad",   "gloomy",    "positive"),  # 近い
]

print("コサイン類似度の動作確認:\n")
for a, b, expected in pairs_to_check:
    sim = cosine_similarity([vec_dict[a]], [vec_dict[b]])[0, 0]
    actual = "positive" if sim > 0 else "negative"
    ok = "✅" if actual == expected else "⚠️"
    print(f"  {ok} {a:10s} <-> {b:10s}: cosine = {sim:+.3f}   (期待: {expected})")

print("\n全行に ✅ が付いていれば、ベクトルは妥当です。")


---

## ステップ 9: GitHub にアップロードする

3 つのファイルを GitHub にアップロードして、学生ノートブックから自動取得できるようにします。

### アップロード手順

1. **Colab 左サイドバーから 3 つのファイルをダウンロード**

   左側のフォルダアイコン → `cache_to_upload/` に下記 3 ファイルがあります:
   - `stories_smoke.jsonl`
   - `activations_smoke.npz`
   - `emotion_vectors_smoke.npz`

   各ファイルを右クリック → 「ダウンロード」

2. **GitHub リポジトリの `data/` フォルダにアップロード**

   - リポジトリ: `https://github.com/CSS-Laboratory/LLMintro`
   - `data/` フォルダ (なければ新規作成) を開く
   - 「Add file」→ 「Upload files」 → 3 ファイルをドラッグ&ドロップ
   - コミットメッセージ例: `Update emotion vector cache (YYYY-MM-DD)`
   - 「Commit changes」

3. **学生ノートブックの動作確認**

   学生ノートブックを開き、`LIVE_MODE = False` にして頭から実行してみる。  
   GitHub から `stories_smoke.jsonl` などが正しくダウンロードされ、最終セクションのヒートマップまで描画されれば成功です。

### ファイルサイズの目安

| ファイル | サイズ目安 |
|---|---|
| `stories_smoke.jsonl` | 約 200-500 KB |
| `activations_smoke.npz` | 約 2-3 MB |
| `emotion_vectors_smoke.npz` | 約 100 KB |

GitHub の単一ファイル上限 (100 MB) には十分余裕があります。

### よくある落とし穴

- **アップロード前に古いキャッシュを消し忘れる** → 新規にプル → 古いデータが混ざる。アップロード時に **置き換え (overwrite)** することを確認してください。
- **GitHub Pages 経由ではなく `raw.githubusercontent.com` を使う** → 学生ノートブックは `raw.githubusercontent.com/CSS-Laboratory/LLMintro/main/data/...` を参照しています。リポジトリのデフォルトブランチが `main` であることを確認してください (`master` の場合は学生ノートブックの `GITHUB_DATA_BASE` URL を変更)。
- **ファイル名のスペル**: 学生ノートブック側で固定されているので、変更する場合は両方一致させてください。

### 本格的な研究用途には?

このノートブックは講義用に **20 ストーリー × 14 感情** の小規模設定です。研究用には:

- `STORIES_PER_TOPIC` と `TOPICS_PER_EMOTION` を増やす (Anthropic の元論文は 1 感情あたり数百ストーリー)
- 中性 (neutral) ダイアログを別途生成して、グローバル中心の代わりに使う
- 複数のシードで再計算し、ベクトルの安定性を確認する
- 複数のレイヤーで同じ実験をして、層ごとの感情表現の違いを比較する

完全版は元の研究ノートブック (`Emotion_Concepts_and_Their_Function.ipynb`) を参照してください。
